# Mistral OCR + Vector Search Cookbook

A comprehensive guide to building an intelligent document processing and search system using:
- **Mistral OCR** for text extraction from images and PDFs
- **FastEmbed** for generating semantic embeddings
- **Qdrant** for vector storage and similarity search

This cookbook demonstrates how to create a complete pipeline for processing handwritten notes, documents, and performing semantic search across them.

## Prerequisites

You'll need:
- Python 3.9+
- Mistral API key (sign up at [Mistral AI](https://mistral.ai/))
- Qdrant instance (cloud or self-hosted)
- Sample images or PDFs to test with

## Installation

Install the required packages:

In [ ]:
!pip install mistralai==1.5.1 fastembed==0.6.0 qdrant-client==1.13.3 python-dotenv==1.0.1

## Environment Setup

Configure your API keys and settings:

In [ ]:
import os
from dotenv import load_dotenv

# Load environment variables
load_dotenv()

# Set your API keys here or in a .env file
MISTRAL_API_KEY = os.getenv("MISTRAL_API_KEY", "your-mistral-api-key")
QDRANT_URL = os.getenv("QDRANT_URL", "your-qdrant-url")
QDRANT_API_KEY = os.getenv("QDRANT_API_KEY", "your-qdrant-api-key")
COLLECTION_NAME = os.getenv("COLLECTION_NAME", "notes")

print("Environment configured!")

## 1. OCR Processing with Mistral

Extract text from images and PDFs using Mistral's OCR API:

In [ ]:
import tempfile
import mimetypes
from mistralai import Mistral

def extract_text_from_file(file_path):
    """
    Extract text from an image or PDF using Mistral OCR API
    """
    if not MISTRAL_API_KEY:
        raise ValueError("Mistral API key not configured")
    
    # Initialize Mistral client
    client = Mistral(api_key=MISTRAL_API_KEY)
    
    # Determine file type
    file_extension = file_path.split('.')[-1].lower()
    
    # Upload the file
    with open(file_path, "rb") as f:
        uploaded_file = client.files.upload(
            file={
                "file_name": file_path.split('/')[-1],
                "content": f,
            },
            purpose="ocr"
        )
    
    # Get the signed URL for the uploaded file
    signed_url = client.files.get_signed_url(file_id=uploaded_file.id)
    
    # Process with OCR
    ocr_model = "mistral-ocr-latest"
    
    if file_extension == 'pdf':
        response = client.ocr.process(
            model=ocr_model,
            document={
                "type": "document_url",
                "document_url": signed_url.url
            }
        )
    else:
        response = client.ocr.process(
            model=ocr_model,
            document={
                "type": "image_url",
                "image_url": signed_url.url
            }
        )
    
    # Extract text from the response
    extracted_text = "\n\n".join([f"### Page {i+1}\n{response.pages[i].markdown}" for i in range(len(response.pages))])
    
    return extracted_text

# Example usage (uncomment and modify path to test)
# text = extract_text_from_file("path/to/your/image_or_pdf.jpg")
# print("Extracted text:")
# print(text[:500] + "..." if len(text) > 500 else text)

## 2. Embedding Generation with FastEmbed

Convert text into semantic vector embeddings:

In [ ]:
from fastembed import TextEmbedding

# Initialize the embedding model
embedder = None

def get_embedder():
    """
    Get or initialize the embedding model
    """
    global embedder
    if embedder is None:
        embedder = TextEmbedding(model_name="BAAI/bge-base-en-v1.5")
    return embedder

def generate_embedding(text):
    """
    Generate embedding for a text
    """
    embedder = get_embedder()
    embedding = list(embedder.embed([text]))[0]
    return embedding.tolist()  # Convert numpy array to list

# Example usage
sample_text = "This is a sample note about machine learning and artificial intelligence."
embedding = generate_embedding(sample_text)
print(f"Generated embedding with {len(embedding)} dimensions")
print(f"First 5 values: {embedding[:5]}")

## 3. Vector Storage with Qdrant

Set up Qdrant for vector storage and retrieval:

In [ ]:
from qdrant_client import QdrantClient
from qdrant_client.http import models
import uuid

# Initialize Qdrant client
qdrant_client = None

def get_qdrant_client():
    """
    Get or initialize the Qdrant client
    """
    global qdrant_client
    if qdrant_client is None:
        if not QDRANT_URL or not QDRANT_API_KEY:
            raise ValueError("Qdrant URL or API key not configured")
        
        qdrant_client = QdrantClient(url=QDRANT_URL, api_key=QDRANT_API_KEY)
        
        # Check if collection exists, if not create it
        collections = qdrant_client.get_collections().collections
        collection_names = [collection.name for collection in collections]
        
        if COLLECTION_NAME not in collection_names:
            qdrant_client.create_collection(
                collection_name=COLLECTION_NAME,
                vectors_config=models.VectorParams(
                    size=768,  # Size of BAAI/bge-base-en-v1.5 embeddings
                    distance=models.Distance.COSINE
                )
            )
            print(f"Created collection: {COLLECTION_NAME}")
        else:
            print(f"Collection {COLLECTION_NAME} already exists")
    
    return qdrant_client

def store_document(text, embedding, doc_type, metadata=None):
    """
    Store a document with its embedding in Qdrant
    """
    client = get_qdrant_client()
    
    # Generate a unique ID
    doc_id = str(uuid.uuid4())
    
    # Prepare metadata
    payload = {
        "text": text,
        "doc_type": doc_type  # "note" or "document"
    }
    if metadata:
        payload.update(metadata)
    
    client.upsert(
        collection_name=COLLECTION_NAME,
        points=[
            models.PointStruct(
                id=doc_id,
                vector=embedding,
                payload=payload
            )
        ]
    )
    return {"id": doc_id, "text": text, "doc_type": doc_type}

# Initialize Qdrant
client = get_qdrant_client()

## 4. Semantic Search Implementation

Search for similar documents using vector similarity:

In [ ]:
def search_documents(query_embedding, target_type=None, limit=5):
    """
    Search for documents using a query embedding
    
    Parameters:
    - query_embedding: The embedding vector to search with
    - target_type: Filter results by document type ("note", "document", or None for both)
    - limit: Maximum number of results to return
    """
    client = get_qdrant_client()
    
    # Prepare filter condition if target type is specified
    filter_condition = None
    if target_type:
        filter_condition = models.Filter(
            must=[
                models.FieldCondition(
                    key="doc_type",
                    match=models.MatchValue(value=target_type)
                )
            ]
        )
    
    results = client.search(
        collection_name=COLLECTION_NAME,
        query_vector=query_embedding,
        limit=limit,
        query_filter=filter_condition
    )
    
    return [
        {
            "id": hit.id,
            "text": hit.payload.get("text", ""),
            "doc_type": hit.payload.get("doc_type", "unknown"),
            "title": hit.payload.get("title", ""),
            "score": hit.score
        }
        for hit in results
    ]

def text_search(query_text, target_type=None, limit=5):
    """
    Perform text-based semantic search
    """
    # Generate embedding for the query
    query_embedding = generate_embedding(query_text)
    
    # Search for similar documents
    return search_documents(query_embedding, target_type, limit)

print("Search functions ready!")

## 5. Complete Example Workflow

Let's walk through a complete example of processing and searching documents:

### Step 1: Process a document with OCR

In [ ]:
# Example: Process an image or PDF
# Replace with your actual file path
file_path = "test.pdf"  # or "your_image.jpg"

try:
    # Extract text using OCR
    extracted_text = extract_text_from_file(file_path)
    print("✅ Text extracted successfully!")
    print(f"📄 Text length: {len(extracted_text)} characters")
    print(f"📄 Preview: {extracted_text[:200]}...")
except Exception as e:
    print(f"❌ OCR Error: {e}")

### Step 2: Generate embeddings and store in vector database

In [ ]:
# Generate embedding for the extracted text
try:
    embedding = generate_embedding(extracted_text)
    print(f"✅ Embedding generated: {len(embedding)} dimensions")
    
    # Store in Qdrant
    result = store_document(
        text=extracted_text,
        embedding=embedding,
        doc_type="document",  # or "note"
        metadata={"title": "Sample Document", "source": file_path}
    )
    
    print(f"✅ Document stored with ID: {result['id']}")
    document_id = result['id']
    
except Exception as e:
    print(f"❌ Storage Error: {e}")

### Step 3: Perform semantic search

In [ ]:
# Search for documents using text queries
search_queries = [
    "machine learning algorithms",
    "data analysis techniques",
    "project management"
]

for query in search_queries:
    print(f"\n🔍 Searching for: '{query}'")
    
    try:
        results = text_search(query, limit=3)
        
        if results:
            for i, result in enumerate(results, 1):
                print(f"  {i}. Score: {result['score']:.3f} | Type: {result['doc_type']}")
                print(f"     Text: {result['text'][:100]}...")
        else:
            print("  No results found")
    except Exception as e:
        print(f"  ❌ Search Error: {e}")

## 6. Advanced Use Cases

### Batch Processing Multiple Files

In [ ]:
import os
import glob

def batch_process_files(directory_path, doc_type="document"):
    """
    Process all images and PDFs in a directory
    """
    # Supported file types
    file_patterns = ['*.jpg', '*.jpeg', '*.png', '*.pdf']
    
    processed_files = []
    
    for pattern in file_patterns:
        files = glob.glob(os.path.join(directory_path, pattern))
        
        for file_path in files:
            try:
                print(f"Processing: {file_path}")
                
                # Extract text
                text = extract_text_from_file(file_path)
                
                # Generate embedding
                embedding = generate_embedding(text)
                
                # Store in database
                result = store_document(
                    text=text,
                    embedding=embedding,
                    doc_type=doc_type,
                    metadata={
                        "title": os.path.basename(file_path),
                        "source": file_path
                    }
                )
                
                processed_files.append(result)
                print(f"✅ Stored: {result['id']}")
                
            except Exception as e:
                print(f"❌ Error processing {file_path}: {e}")
    
    return processed_files

# Example usage (uncomment to test)
# processed = batch_process_files("./sample_documents/", doc_type="reference")
# print(f"\nProcessed {len(processed)} files")

### Search by Image

Upload an image and find similar content:

In [ ]:
def search_by_image(image_path, limit=5):
    """
    Extract text from an image and search for similar documents
    """
    try:
        # Extract text from the search image
        search_text = extract_text_from_file(image_path)
        print(f"📄 Extracted search text: {search_text[:100]}...")
        
        # Search for similar documents
        results = text_search(search_text, limit=limit)
        
        return results
        
    except Exception as e:
        print(f"❌ Image search error: {e}")
        return []

# Example usage (uncomment to test)
# results = search_by_image("query_image.jpg")
# for i, result in enumerate(results, 1):
#     print(f"{i}. {result['title']} (Score: {result['score']:.3f})")

## 7. Document Type Classification

Organize your content by classifying documents into different types:

In [ ]:
from enum import Enum

class DocType(str, Enum):
    NOTE = "note"           # Personal notes, meeting notes, to-do lists
    DOCUMENT = "document"   # Reference materials, textbooks, articles

def classify_and_store(file_path, doc_type, title=None):
    """
    Complete pipeline: OCR -> Embedding -> Storage with classification
    """
    try:
        # 1. Extract text with OCR
        text = extract_text_from_file(file_path)
        
        # 2. Generate embedding
        embedding = generate_embedding(text)
        
        # 3. Store with classification
        metadata = {"title": title or os.path.basename(file_path)}
        result = store_document(text, embedding, doc_type, metadata)
        
        print(f"✅ {doc_type.capitalize()} processed and stored: {result['id']}")
        return result
        
    except Exception as e:
        print(f"❌ Pipeline error: {e}")
        return None

# Example usage
# Classify different types of documents
examples = [
    ("meeting_notes.jpg", DocType.NOTE, "Weekly Team Meeting"),
    ("research_paper.pdf", DocType.DOCUMENT, "ML Research Paper"),
    ("todo_list.png", DocType.NOTE, "Project Tasks")
]

# Process examples (uncomment when you have files)
# for file_path, doc_type, title in examples:
#     if os.path.exists(file_path):
#         classify_and_store(file_path, doc_type, title)

## 8. Advanced Search Scenarios

### Filter by Document Type

In [ ]:
# Search only in personal notes
query = "project deadline"
note_results = text_search(query, target_type="note", limit=3)
print(f"📝 Notes about '{query}':")
for result in note_results:
    print(f"  - {result['title']} (Score: {result['score']:.3f})")

print()

# Search only in reference documents
doc_results = text_search(query, target_type="document", limit=3)
print(f"📚 Documents about '{query}':")
for result in doc_results:
    print(f"  - {result['title']} (Score: {result['score']:.3f})")

### Find Similar Documents

In [ ]:
def find_similar_documents(document_id, limit=5):
    """
    Find documents similar to a specific document
    """
    client = get_qdrant_client()
    
    # Get the source document
    points = client.retrieve(
        collection_name=COLLECTION_NAME,
        ids=[document_id],
        with_vectors=True
    )
    
    if not points:
        print(f"Document {document_id} not found")
        return []
    
    source_doc = points[0]
    
    # Search for similar documents
    results = search_documents(source_doc.vector, limit=limit+1)
    
    # Filter out the source document
    similar_docs = [r for r in results if r["id"] != document_id][:limit]
    
    return similar_docs

# Example usage (uncomment when you have stored documents)
# if 'document_id' in locals():
#     similar = find_similar_documents(document_id, limit=3)
#     print(f"📄 Documents similar to {document_id}:")
#     for doc in similar:
#         print(f"  - {doc['title']} (Score: {doc['score']:.3f})")

## 9. Practical Examples

### Personal Note Management System

In [ ]:
class NoteManager:
    def __init__(self):
        self.client = get_qdrant_client()
        self.embedder = get_embedder()
    
    def add_note(self, file_path, title=None, tags=None):
        """Add a handwritten note to the system"""
        text = extract_text_from_file(file_path)
        embedding = generate_embedding(text)
        
        metadata = {
            "title": title or os.path.basename(file_path),
            "tags": tags or [],
            "source_file": file_path
        }
        
        return store_document(text, embedding, "note", metadata)
    
    def search_notes(self, query, limit=5):
        """Search through personal notes"""
        return text_search(query, target_type="note", limit=limit)
    
    def add_reference(self, file_path, title=None, category=None):
        """Add a reference document to the system"""
        text = extract_text_from_file(file_path)
        embedding = generate_embedding(text)
        
        metadata = {
            "title": title or os.path.basename(file_path),
            "category": category,
            "source_file": file_path
        }
        
        return store_document(text, embedding, "document", metadata)

# Initialize note manager
note_manager = NoteManager()
print("📝 Note Manager initialized!")

### Research Assistant Workflow

In [ ]:
def research_workflow(topic, note_files=None, reference_files=None):
    """
    Complete research workflow: process notes and references, then search
    """
    print(f"🔬 Starting research on: {topic}")
    
    # Process personal notes
    if note_files:
        print("\n📝 Processing personal notes...")
        for note_file in note_files:
            if os.path.exists(note_file):
                note_manager.add_note(note_file, title=f"Notes on {topic}")
    
    # Process reference materials
    if reference_files:
        print("\n📚 Processing reference materials...")
        for ref_file in reference_files:
            if os.path.exists(ref_file):
                note_manager.add_reference(ref_file, category=topic)
    
    # Search across all content
    print(f"\n🔍 Searching for content related to '{topic}'...")
    results = text_search(topic, limit=5)
    
    for i, result in enumerate(results, 1):
        print(f"\n{i}. {result['title']} ({result['doc_type']})")
        print(f"   Score: {result['score']:.3f}")
        print(f"   Preview: {result['text'][:150]}...")
    
    return results

# Example research workflow
# results = research_workflow(
#     topic="deep learning",
#     note_files=["class_notes.jpg", "research_ideas.png"],
#     reference_files=["deep_learning_paper.pdf"]
# )

## 10. Performance Tips & Best Practices

### Optimizing OCR Quality

In [ ]:
# Tips for better OCR results
print("""
🎯 OCR Best Practices:

Image Quality:
• Use high-resolution images (300+ DPI)
• Ensure good lighting and contrast
• Keep text horizontal and avoid skew
• Crop to focus on text areas

File Formats:
• PNG/JPG for images
• PDF for multi-page documents
• Avoid heavily compressed images

Content Organization:
• Use descriptive titles and metadata
• Consistent document types (note vs document)
• Add relevant tags for better categorization
""")

### Embedding and Search Optimization

In [ ]:
# Performance monitoring
def analyze_search_performance(queries):
    """
    Analyze search performance across multiple queries
    """
    import time
    
    results = []
    
    for query in queries:
        start_time = time.time()
        
        # Perform search
        search_results = text_search(query, limit=5)
        
        end_time = time.time()
        
        results.append({
            "query": query,
            "num_results": len(search_results),
            "time_ms": (end_time - start_time) * 1000,
            "avg_score": sum(r['score'] for r in search_results) / len(search_results) if search_results else 0
        })
    
    return results

# Example performance analysis
test_queries = ["machine learning", "project management", "data analysis"]
# performance = analyze_search_performance(test_queries)
# for result in performance:
#     print(f"'{result['query']}': {result['num_results']} results in {result['time_ms']:.1f}ms (avg score: {result['avg_score']:.3f})")

## 11. Integration Examples

### Web Application Integration

This cookbook is based on a complete web application. Check out the full implementation in the repository:

- **Frontend**: Next.js with TypeScript (`app/` directory)
- **Backend**: FastAPI (`api/` directory)
- **Deployment**: Configured for Vercel + Fly.io

The application demonstrates:
- File upload interface
- Real-time search
- Document similarity browsing
- Responsive design with Tailwind CSS

## Troubleshooting

### Common Issues

**OCR Processing:**
- Ensure Mistral API key has sufficient credits
- Check file formats are supported (JPG, PNG, PDF)
- Verify file size limits

**Embeddings:**
- FastEmbed model downloads automatically on first use
- Ensure sufficient disk space for model cache

**Vector Search:**
- Verify Qdrant instance is accessible
- Check collection exists and has correct dimensions (768)
- Ensure API keys have proper permissions

### Error Handling Example

In [ ]:
def robust_document_processing(file_path, doc_type, title=None, max_retries=3):
    """
    Robust document processing with error handling and retries
    """
    for attempt in range(max_retries):
        try:
            result = classify_and_store(file_path, doc_type, title)
            return result
        except Exception as e:
            print(f"Attempt {attempt + 1} failed: {e}")
            if attempt == max_retries - 1:
                print(f"❌ Failed to process {file_path} after {max_retries} attempts")
                return None
            
            # Wait before retry
            import time
            time.sleep(2 ** attempt)  # Exponential backoff

print("🛠️ Robust processing functions ready!")

## Next Steps

This cookbook provides a foundation for building intelligent document processing systems. You can extend it by:

1. **Adding more document types**: Spreadsheets, presentations, etc.
2. **Implementing custom embeddings**: Fine-tune models for your domain
3. **Building specialized search**: Add filters, facets, and advanced queries
4. **Creating web interfaces**: Build user-friendly applications
5. **Scaling for production**: Add caching, batch processing, and monitoring

### Related Resources

- [Mistral OCR Documentation](https://docs.mistral.ai/capabilities/ocr/)
- [FastEmbed Models](https://qdrant.github.io/fastembed/)
- [Qdrant Vector Database](https://qdrant.tech/documentation/)
- [Complete Application Code](../)

Happy building! 🚀